# Notebook for creating NGBoost models

Feature selection is (somewhat) done in the feature selection notebook. However the only take away I could get was that having correlated features decreased model performance and than having a lot of features causes overfitting. The final features can be found in the "utils/fetch_data_for_NGBoost" 

On how to use NGBoost
https://stanfordmlgroup.github.io/ngboost/intro.html


In [1]:
import sys
import os
from ngboost import NGBRegressor
from sklearn.metrics import mean_squared_error
from ngboost.distns import LogNormal

# Add project root to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
from utils.fetch_data_for_NGBoost.fwd import fetch_fwd_data_for_NGBoost
from utils.fetch_data_for_NGBoost.mid import fetch_mid_data_for_NGBoost
from utils.fetch_data_for_NGBoost.def_ import fetch_def_data_for_NGBoost
from utils.fetch_data_for_NGBoost.gk import fetch_gk_data_for_NGBoost


## FWD

In [ ]:
X_train = fetch_fwd_data_for_NGBoost(fetch_test_set=False)

y_train = X_train['total_points']
X_train = X_train.drop(columns=['total_points'])

X_test = fetch_fwd_data_for_NGBoost(fetch_test_set=True)
y_test = X_test['total_points']
X_test = X_test.drop(columns=['total_points'])


ngb_fwd = NGBRegressor(verbose=True, n_estimators=500, learning_rate=0.01, 
                       natural_gradient=True, minibatch_frac=0.1,
                       verbose_eval=100)

print("--- Training NGBoost Model for FWD ---")
ngb_fwd.fit(X_train, y_train)
print("--- Training Complete ---")

# Make predictions on the test set
y_preds = ngb_fwd.predict(X_test)
y_dists = ngb_fwd.pred_dist(X_test)

# Evaluate the model's performance
test_MSE = mean_squared_error(y_test, y_preds)

# Test Negative Log Likelihood (how well the predicted distributions match the actual outcomes)
test_NLL = -y_dists.logpdf(y_test.values).mean()

# 5. Print the evaluation metrics
print(f"\n--- FWD Model Evaluation ---")
print(f'Test MSE: {test_MSE:.4f}')
print(f'Test NLL: {test_NLL:.4f}')

# You can also inspect the parameters of the predicted distributions (mean and standard deviation)
print("\nPredicted distribution parameters for the first 5 test samples:")
print(y_dists[0:5].params)

del X_train, y_train, X_test, y_test, y_preds, y_dists

Creating hybrid feature set for FWD
Num elements in FWD data before filter: 6967
Num elements in FWD data after filter: 2922
Creating features for predicting the variance (volatility, uncertainty)...
Creating EWMA features...
Improved Hybrid feature set created successfully
Creating hybrid feature set for FWD


c:\Users\trygt\fpl_dir\fpl_team_selector\utils\fetch_data_for_NGBoost\fwd.py:19: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/2024-25/gws/merged_gw.csv", usecols=fwd_features)


Num elements in FWD data before filter: 1562
Num elements in FWD data after filter: 723
Creating features for predicting the variance (volatility, uncertainty)...
Creating EWMA features...
Improved Hybrid feature set created successfully
--- Training NGBoost Model for FWD ---
[iter 0] loss=2.6303 val_loss=0.0000 scale=1.0000 norm=2.6513
[iter 100] loss=2.4306 val_loss=0.0000 scale=1.0000 norm=2.2408
[iter 200] loss=2.4491 val_loss=0.0000 scale=1.0000 norm=2.3949
[iter 300] loss=2.3019 val_loss=0.0000 scale=1.0000 norm=2.1313
[iter 400] loss=2.2640 val_loss=0.0000 scale=2.0000 norm=4.0529
--- Training Complete ---

--- FWD Model Evaluation ---
Test MSE: 11.8266
Test NLL: 2.7439

Predicted distribution parameters for the first 5 test samples:
{'loc': array([2.07150609, 5.69896861, 2.55853102, 2.40520628, 2.09642319]), 'scale': array([5.00430002, 5.9688738 , 1.88375243, 2.08162978, 2.38675246])}
